# Compiling C code ARMv7 reference

In [1]:
%%bash
cd ./armv7
gcc -fPIC -shared -O3 *.c -o libascon.so

# Importing Libraries

In [2]:
import pynq
from pynq import Overlay, allocate
import numpy as np
import time
import ctypes
import os
import pandas as pd

# Configurations and AXI Memory Mapping

In [3]:
FREQ_CPU = 650e6  
FPGA_CLOCK_FREQ = 100e6 
ITERATIONS = 1000

REG_OFFSETS_ENC = {
    'c'    : 0x18,
    'clen' : 0x24,
    'm'    : 0x3c,
    'mlen' : 0x48, 
    'ad'   : 0x54,
    'adlen': 0x60,
    'nsec' : 0x6c, 
    'npub' : 0x78, 
    'k'    : 0x84 
}

REG_OFFSETS_DEC = {
    'm'    : 0x18,
    'mlen' : 0x24,
    'nsec' : 0x3c,
    'c'    : 0x48, 
    'clen' : 0x54,
    'ad'   : 0x60, 
    'adlen': 0x6c,
    'npub' : 0x78, 
    'k'    : 0x84 
}

# Loading software shared library

In [4]:
lib_path = os.path.abspath("./armv7/libascon.so")
ascon_cpu = ctypes.CDLL(lib_path)

ascon_cpu.crypto_aead_encrypt.argtypes = [
    ctypes.POINTER(ctypes.c_ubyte), ctypes.POINTER(ctypes.c_ulonglong),
    ctypes.POINTER(ctypes.c_ubyte), ctypes.c_ulonglong,
    ctypes.POINTER(ctypes.c_ubyte), ctypes.c_ulonglong,
    ctypes.POINTER(ctypes.c_ubyte), ctypes.POINTER(ctypes.c_ubyte),
    ctypes.POINTER(ctypes.c_ubyte)
]
ascon_cpu.crypto_aead_decrypt.argtypes = [
    ctypes.POINTER(ctypes.c_ubyte), ctypes.POINTER(ctypes.c_ulonglong),
    ctypes.POINTER(ctypes.c_ubyte), ctypes.POINTER(ctypes.c_ubyte),
    ctypes.c_ulonglong, ctypes.POINTER(ctypes.c_ubyte),
    ctypes.c_ulonglong, ctypes.POINTER(ctypes.c_ubyte),
    ctypes.POINTER(ctypes.c_ubyte)
]

# Loading Overlay and setting test parameters

In [5]:
print("Loading encryption bitstream")
overlay_enc = Overlay("./bitstream_files/aead_enc.bit")
ip_enc = overlay_enc.crypto_aead_encrypt_0

MSG_LEN = 1024 
AD_LEN  = 32  
KEY_LEN = 16   
PUB_LEN = 16  
TAG_LEN = 16 

# Memory allocation
m_buf     = allocate(shape=(MSG_LEN,), dtype=np.uint8)
ad_buf    = allocate(shape=(AD_LEN,), dtype=np.uint8)
k_buf     = allocate(shape=(KEY_LEN,), dtype=np.uint8)
npub_buf  = allocate(shape=(PUB_LEN,), dtype=np.uint8)
nsec_buf  = allocate(shape=(1,), dtype=np.uint8) 

# Output buffers (FPGA)
c_buf_hw     = allocate(shape=(MSG_LEN + TAG_LEN,), dtype=np.uint8)
clen_buf_hw  = allocate(shape=(1,), dtype=np.uint64)
m_out_hw     = allocate(shape=(MSG_LEN,), dtype=np.uint8)
mlen_buf_hw  = allocate(shape=(1,), dtype=np.uint64)

# Output buffers (CPU)
c_buf_sw     = (ctypes.c_ubyte * (MSG_LEN + TAG_LEN))()
clen_buf_sw  = ctypes.c_ulonglong(0)
m_out_sw     = (ctypes.c_ubyte * MSG_LEN)()
mlen_buf_sw  = ctypes.c_ulonglong(0)

# Filling buffers with random data
m_buf[:]    = np.random.randint(0, 256, MSG_LEN, dtype=np.uint8)
ad_buf[:]   = np.random.randint(0, 256, AD_LEN, dtype=np.uint8)
k_buf[:]    = np.random.randint(0, 256, KEY_LEN, dtype=np.uint8)
npub_buf[:] = np.random.randint(0, 256, PUB_LEN, dtype=np.uint8)

Loading encryption bitstream


# Auxiliary functions for FPGA execution

In [6]:
def run_fpga_ip(ip, offsets, kwargs_dict, iterations):
    for arg_name, value in kwargs_dict.items():
        if isinstance(value, pynq.buffer.PynqBuffer):
            ip.write(offsets[arg_name], value.device_address)
        else:
            ip.write(offsets[arg_name], value)
            
    total_time = 0.0
    for _ in range(iterations):
        start_time = time.time()
        ip.write(0x00, 0x01)
        
        while not (ip.read(0x00) & 0x02):
            pass
        end_time = time.time()
        total_time += (end_time - start_time)
    
    return total_time / iterations

# Encryption Test

In [7]:
print(f"Testing encryption on CPU ({ITERATIONS} iterations)")
total_cpu_enc_time = 0.0
for _ in range(ITERATIONS):
    start_time = time.time()
    ascon_cpu.crypto_aead_encrypt(
        c_buf_sw, ctypes.byref(clen_buf_sw),
        m_buf.ctypes.data_as(ctypes.POINTER(ctypes.c_ubyte)), MSG_LEN,
        ad_buf.ctypes.data_as(ctypes.POINTER(ctypes.c_ubyte)), AD_LEN,
        None, npub_buf.ctypes.data_as(ctypes.POINTER(ctypes.c_ubyte)),
        k_buf.ctypes.data_as(ctypes.POINTER(ctypes.c_ubyte))
    )
    total_cpu_enc_time += (time.time() - start_time)

cpu_enc_time = total_cpu_enc_time / ITERATIONS

print(f"Testing encryption on FPGA ({ITERATIONS} iterations)")
fpga_enc_time = run_fpga_ip(ip_enc, REG_OFFSETS_ENC, {
    'c': c_buf_hw, 'clen': clen_buf_hw, 'm': m_buf, 'mlen': MSG_LEN,
    'ad': ad_buf, 'adlen': AD_LEN, 'nsec': nsec_buf, 'npub': npub_buf, 'k': k_buf
}, iterations=ITERATIONS)

c_hw = np.array(c_buf_hw[:clen_buf_sw.value])
c_sw = np.array(c_buf_sw)[:clen_buf_sw.value]
enc_match = np.array_equal(c_hw, c_sw)
print(f"Ciphertext FPGA == CPU? {enc_match}")

Testing encryption on CPU (1000 iterations)
Testing encryption on FPGA (1000 iterations)
Ciphertext FPGA == CPU? True


In [8]:
print("\nFreeing the encryption IP and loading the decryption IP")
overlay_enc.free() 
overlay_dec = Overlay("./bitstream_files/aead_dec.bit")
ip_dec = overlay_dec.crypto_aead_decrypt_0

print(f"Executing decryption on CPU ({ITERATIONS} iterations)")
total_cpu_dec_time = 0.0
for _ in range(ITERATIONS):
    start_time = time.time()
    ascon_cpu.crypto_aead_decrypt(
        m_out_sw, ctypes.byref(mlen_buf_sw), None,
        c_buf_sw, clen_buf_sw.value,
        ad_buf.ctypes.data_as(ctypes.POINTER(ctypes.c_ubyte)), AD_LEN,
        npub_buf.ctypes.data_as(ctypes.POINTER(ctypes.c_ubyte)),
        k_buf.ctypes.data_as(ctypes.POINTER(ctypes.c_ubyte))
    )
    total_cpu_dec_time += (time.time() - start_time)

cpu_dec_time = total_cpu_dec_time / ITERATIONS

print(f"Executing decryption on FPGA ({ITERATIONS} iterations)")
fpga_dec_time = run_fpga_ip(ip_dec, REG_OFFSETS_DEC, {
    'm': m_out_hw, 'mlen': mlen_buf_hw, 'nsec': nsec_buf,
    'c': c_buf_hw, 'clen': clen_buf_sw.value,
    'ad': ad_buf, 'adlen': AD_LEN, 'npub': npub_buf, 'k': k_buf
}, iterations=ITERATIONS)

m_hw = np.array(m_out_hw[:MSG_LEN])
dec_match = np.array_equal(m_hw, m_buf)
print(f"Plaintext recovered successfully? {dec_match}")


Freeing the encryption IP and loading the decryption IP
Executing decryption on CPU (1000 iterations)
Executing decryption on FPGA (1000 iterations)
Plaintext recovered successfully? True


# Results

In [9]:
fpga_enc_cycles = int(fpga_enc_time * FPGA_CLOCK_FREQ)
fpga_dec_cycles = int(fpga_dec_time * FPGA_CLOCK_FREQ)
cpu_enc_cycles = int(cpu_enc_time * FREQ_CPU)
cpu_dec_cycles = int(cpu_dec_time * FREQ_CPU) 

data = {
    "Platform": ["Avg CPU (Cortex-A9)", "Avg FPGA (PYNQ-Z2)", "Speedup (FPGA/CPU)"],
    "Encryption Time (ms)": [cpu_enc_time*1000, fpga_enc_time*1000, cpu_enc_time/fpga_enc_time],
    "Encryption Cycles": [cpu_enc_cycles, fpga_enc_cycles, "-"],
    "Time Decryption (ms)": [cpu_dec_time*1000, fpga_dec_time*1000, cpu_dec_time/fpga_dec_time],
    "Decryption Cycles": [cpu_dec_cycles, fpga_dec_cycles, "-"],
    "Hardware Correctness": ["-", "Pass" if enc_match and dec_match else "Fail", "-"]
}

df_metrics = pd.DataFrame(data)
display(df_metrics)

# Memory free
m_buf.freebuffer()
ad_buf.freebuffer()
k_buf.freebuffer()
npub_buf.freebuffer()
nsec_buf.freebuffer()
c_buf_hw.freebuffer()
clen_buf_hw.freebuffer()
m_out_hw.freebuffer()
mlen_buf_hw.freebuffer()
overlay_dec.free()

,Platform,Encryption Time (ms),Encryption Cycles,Time Decryption (ms),Decryption Cycles,Hardware Correctness
0,Avg CPU (Cortex-A9),0.419090,272408,0.357513,232383,-
1,Avg FPGA (PYNQ-Z2),0.066079,6607,0.063807,6380,Pass
2,Speedup (FPGA/CPU),6.342293,-,5.603081,-,-
